# Stage 1: Non-Instruction Domain Fine-Tuning
This notebook adapts a base language model to the **Practice Your Speech** domain using raw domain text. The model learns core terminology, heuristics, and technical guidelines via next-token prediction prior to learning Q&A behaviors.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Install Unsloth and required libraries for Google Colab/Kaggle
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl peft transformers accelerate bitsandbytes
!pip install unsloth_zoo
!pip install datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-eupefs6v/unsloth_5b2ba8623a184672ac46d1f2b35044e7
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-eupefs6v/unsloth_5b2ba8623a184672ac46d1f2b35044e7
  Resolved https://github.com/unslothai/unsloth.git to commit d105bd7b42ea8d4ecdb3e3364abb605b558c017e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 125.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 25.6 MB/s eta 0:00:00
   

In [3]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None # Auto-detect. Float16 for Tesla T4/V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to run within 16GB GPU memory

# Load Qwen2.5-7B base model as starting point
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [4]:
# Configure LoRA (Low-Rank Adaptation) targets
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank dimension
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.05, # LoRA dropout
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.2 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [5]:
from datasets import load_dataset

# Load the raw domain knowledge text file from Google Drive
dataset = load_dataset("text", data_files={"train": "/content/drive/MyDrive/AIML-2026/non_instruction_data.txt"})

# Map formatting to raw text chunks for pretraining
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=max_seq_length)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/102 [00:00<?, ? examples/s]

In [6]:
tokenized_dataset["train"]

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 102
})

In [7]:
print("First 3 examples from the tokenized dataset:")
for i in range(3):
    print(f"--- Example {i+1} ---")
    print(f"Text: {tokenized_dataset['train'][i]['text']}")
    print(f"Input IDs: {tokenizer.decode(tokenized_dataset['train'][i]['input_ids'])}")
    print(f"Attention Mask: {tokenized_dataset['train'][i]['attention_mask']}")


First 3 examples from the tokenized dataset:
--- Example 1 ---
Text: Practice Your Speech (practiceyourspeech.com) is an advanced, AI-driven software-as-a-service (SaaS) platform designed to evaluate and improve public speaking and speech delivery. By combining audio transcript extraction, video-based facial emotion detection, and high-level large language model (LLM) processing, the platform simulates the feedback of a professional Toastmaster evaluator. It serves four distinct client personas: independent public speakers, students, educators/coaches, and speech competition organizers.
Input IDs: Practice Your Speech (practiceyourspeech.com) is an advanced, AI-driven software-as-a-service (SaaS) platform designed to evaluate and improve public speaking and speech delivery. By combining audio transcript extraction, video-based facial emotion detection, and high-level large language model (LLM) processing, the platform simulates the feedback of a professional Toastmaster evaluator. It s

In [8]:
import torch

# The 'text' column is already removed or not present.
# We will ensure input_ids and attention_mask are torch.long directly when accessed by the trainer.

# Reset format to None to prevent default datasets tensorization that might use torchvision
tokenized_dataset.set_format(type=None)

# Verify the features of the tokenized dataset (should still show List(Value('int32')) and List(Value('int8')))
print(tokenized_dataset["train"].features)

# The DataCollatorForLanguageModeling will handle converting lists of ints to torch.long tensors during batching.
# Explicit map and set_format calls that caused issues are removed.

{'text': Value('string'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}


The `IndexError` indicates that there are empty examples in the `tokenized_dataset`. This can happen if the original text file contains blank lines or very short texts that become empty after tokenization. We need to filter these out to ensure all training examples have valid `input_ids` and `attention_mask`.

In [9]:
# Filter out examples where input_ids or attention_mask are empty
original_num_rows = tokenized_dataset["train"].num_rows

# Filter based on the length of input_ids
filtered_dataset = tokenized_dataset["train"].filter(lambda example: len(example["input_ids"]) > 0)

# Update the train_dataset to the filtered version
tokenized_dataset["train"] = filtered_dataset

print(f"Original number of examples: {original_num_rows}")
print(f"Number of examples after filtering empty ones: {tokenized_dataset['train'].num_rows}")

# Verify the features again after filtering (should be unchanged, but good practice)
print(tokenized_dataset["train"].features)

Filter:   0%|          | 0/102 [00:00<?, ? examples/s]

Original number of examples: 102
Number of examples after filtering empty ones: 51
{'text': Value('string'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}


Now that the empty examples have been removed, you can retry running the training cell (Cell 7).

In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForLanguageModeling

# Initialize DataCollator for Causal Language Modeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Initialize Trainer for Causal Language Modeling / Pretraining
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = tokenized_dataset["train"],
    # dataset_text_field = "text", # This line is removed as the dataset is already tokenized
    max_seq_length = max_seq_length,
    dataset_num_proc = None, # Changed from 2 to None to avoid PicklingError
    packing = False, # Can pack multiple short paragraphs to save memory
    data_collator = data_collator, # Use the custom data collator
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no", # Disable saving to bypass pickling issues during checkpoint creation
        report_to = "none", # Disable reporting to avoid potential issues with logger pickling
    ),
)

# Execute pretraining
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51 | Num Epochs = 15 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,3.544099
10,3.323560
15,2.867754
20,2.641367
25,2.294722
30,2.007796
35,1.667502
40,1.439585
45,1.162572
50,0.863946


In [11]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(
    ["Practice Your Speech calculates Words Per Minute (WPM) by taking"],
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 60, use_cache = True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=60) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

Practice Your Speech calculates Words Per Minute (WPM) by taking the number of words in the Toastmaster Evaluation Report and dividing it by the duration of the video. It also checks if the WPM is below the target rate of 140 WPM. If the WPM is low, the system recommends increasing vocabulary and reducing filler words. The AI also


In [12]:
# Save the adapter locally for Stage 2
model.save_pretrained("non_instruction_lora_model")
tokenizer.save_pretrained("non_instruction_lora_model")

Unsloth: Restored added_tokens_decoder metadata in non_instruction_lora_model/tokenizer_config.json.


('non_instruction_lora_model/tokenizer_config.json',
 'non_instruction_lora_model/tokenizer.json')

In [13]:
# Load adapter weights in 16-bit
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "non_instruction_lora_model",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = False, # Must be False to merge
)
# Save the merged model
model.save_pretrained_merged("/content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage1-merged", tokenizer, save_method="merged_16bit")

==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage1-merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage1-merged`: 100%|██████████| 1/1 [00:58<00:00, 58.38s/it]


Successfully copied all 1 files from cache to `/content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage1-merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:20<00:00, 140.12s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage1-merged`
